# Lesson 10 - AI Agents in Production

In this lesson you will learn **उत्पादनातील नमुने** for AI agents using the Microsoft Agent Framework (Python). We cover:

- **निरिक्षणक्षमता** — एजंट परस्परसंवादांमध्ये वेळ मोजणी आणि लॉगिंग जोडणे
- **मूल्यमापन** — प्रतिसादाच्या गुणवत्तेचे स्कोअर करण्यासाठी एक मूल्यांकन एजंट वापरणे
- **खर्च व्यवस्थापन** — टोकन ऑप्टिमायझेशन आणि मॉडेल निवडीसाठीच्या रणनीती

The scenario is a **ट्रॅव्हल एजंट** that helps users plan trips, with monitoring and evaluation layered on top.


## स्थापना


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## उत्पादन विचार

Moving AI agents from prototypes to production requires careful attention to three pillars:

1. **एनव्हरव्हेबिलिटी (Observability)** — तुम्हाला एजंट काय करतो, त्यास किती वेळ लागतो, आणि तो कोणती साधने कॉल करतो हे पाहण्याची पारदर्शकता आवश्यक आहे. ट्रेसिंग आणि लॉगिंग शिवाय उत्पादनातील समस्यांचे डीबगिंग जवळजवळ अशक्य असते.

2. **मूल्यांकन (Evaluation)** — स्वयंचलित गुणवत्ता तपासण्या हे सुनिश्चित करतात की एजंटची उत्तरे कालांतराने अचूक, पूर्ण आणि उपयुक्त राहतील. एक मूल्यांकन एजंट निर्धारित निकषांनुसार उत्तरांचे गुणांकन करू शकतो.

3. **खर्च व्यवस्थापन (Cost Management)** — टोकनच्या वापरामुळे थेट खर्चावर परिणाम होतो. प्रॉम्प्ट ऑप्टिमायझेशन, मॉडेल निवड आणि कॅशिंगसारख्या धोरणांमुळे गुणवत्ता न बिघडवता खर्च नियंत्रणात ठेवता येतो.


## प्रेक्षणीय एजंट तयार करणे

आम्ही प्रवास साधने परिभाषित करतो आणि एजंट कॉलला टाइमिंगसह आवरण करतो जेणेकरून आपण विलंबता निरीक्षण करू शकू. उत्पादनात तुम्ही OpenTelemetry किंवा तत्सम ट्रेसिंग बॅकएंडसह समाकलित कराल.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन पद्धती

सामान्य उत्पादन पॅटर्न म्हणजे दुसऱ्या एजंटचा उपयोग एक **मूल्यांकनकर्ता** म्हणून करणे. मूल्यांकनकर्ता प्राथमिक एजंटच्या प्रतिसादाला पूर्वनिर्धारित निकषांनुसार गुण देतो, जसे की पूर्णता, अचूकता आणि उपयुक्तता.

हे सक्षम करते:
- वापरकर्त्यांपर्यंत प्रतिसाद पोहोचण्यापूर्वी स्वयंचलित गुणवत्ता नियंत्रण
- प्रॉम्प्ट्स किंवा मॉडेल बदलल्यास रिग्रेशन शोधणे
- एजंटच्या कामगिरीचे कालांतराने निरंतर निरीक्षण


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## खर्च व्यवस्थापन रणनीती

खर्च नियंत्रण करणे उत्पादनात्मक AI एजंट्ससाठी अत्यंत महत्त्वाचे आहे. येथे मुख्य रणनीती आहेत:

| Strategy | Description |
|---|---|
| **प्रॉम्प्ट ऑप्टिमायझेशन** | सिस्टम निर्देश संक्षिप्त ठेवा. इनपुट टोकन कमी करण्यासाठी अनावश्यक संदर्भ काढा. |
| **मॉडेल निवड** | साध्या कामांसाठी जसे वर्गीकरण किंवा एक्स्ट्रॅक्शनसाठी लहान, स्वस्त मॉडेल वापरा (उदा. GPT-4o-mini), आणि जटील तर्कमथनासाठी मोठी मॉडेल राखून ठेवा. |
| **कॅशिंग** | उपकरणांचे परिणाम आणि वारंवार विचारल्या जाणाऱ्या क्वेरी कॅश करा जेणेकरून अनावश्यक API कॉल टाळता येतील. |
| **टोकन बजेट** | `max_tokens` मर्यादा सेट करा जेणेकरून अप्रत्याशितरीत्या लांब प्रतिसाद होणार नाहीत. |
| **बॅचिंग** | परिस्थितीनुसार, एकापेक्षा जास्त वापरकर्ता क्वेरीज शक्य असल्यास एका API कॉलमध्ये गटबद्ध करा. |

व्यवहारात, स्तरबद्ध पद्धत प्रभावी आहे: सोप्या विनंत्या वेगवान, स्वस्त मॉडेलकडे मार्गदर्शित करा आणि फक्त जटिल क्वेरीजच अधिक सामर्थ्यवान मॉडेलकडे अग्रेषित करा.


## Summary

In this lesson you learned how to:

1. **पर्यवेक्षण जोडा** एजंट परस्परसंवादांमध्ये टाइमिंग आणि लॉगिंग वापरून, ट्रेसिंग आणि मॉनिटरिंगसाठी पाया घाला.
2. **एजंटच्या प्रतिसादांचे मूल्यांकन करा** पूर्णता, अचूकता आणि उपयुक्तता यांना गुण देणाऱ्या एक मूल्यांकन एजंटचा वापर करून स्वयंचलितपणे.
3. **खर्च व्यवस्थापित करा** प्रॉम्प्ट ऑप्टिमायझेशन, मॉडेल निवड, कॅशिंग, आणि टोकन बजेटद्वारे.

हे उत्पादन पॅटर्न तुमच्या AI एजंटना प्रमाणावर विश्वसनीय, मोजता येण्याजोगे आणि खर्च-प्रभावी बनवायला मदत करतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
हा दस्तऐवज AI अनुवाद सेवा Co‑op Translator (https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, परंतु कृपया लक्षात ठेवा की स्वयंचलित अनुवादांमध्ये चुका किंवा अचूकतेची कमतरता असू शकते. मूळ भाषेतील दस्तऐवज अधिकृत स्रोत म्हणून मानला पाहिजे. महत्त्वाच्या माहितीसाठी व्यावसायिक मानवी अनुवाद करण्याची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुतींबद्दल किंवा चुकीच्या अर्थलावणीबद्दल आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
